In [3]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

/opt/homebrew/anaconda3/envs/ml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df = pd.read_csv("PolicyPulse_Cleaned.csv")

In [5]:
embeddings = np.load("scheme_embeddings.npy")

print(embeddings.shape)

(3356, 384)


In [6]:
# Create FAISS Index
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 3356


In [7]:
# Save FAISS Index
faiss.write_index(index, "PolicyPulse.index")

In [8]:
# Load FAISS Index
index = faiss.read_index("PolicyPulse.index")

print(index.ntotal)

3356


In [9]:
# Load Embedding Model
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6164.75it/s]


In [27]:
# User query
query = "I am a farmer from Maharashtra looking for financial assistance."

# Query 1
query = "I am a widow living in Maharashtra. I need financial support and pension schemes."

# Query 2
query = "I am a student from a low-income family looking for scholarship schemes for higher education."

# Query 3
query = "I want to start a small business. Are there any government schemes for MSMEs or entrepreneurs?"

# Query 4
query = "I am a differently abled person looking for financial assistance and healthcare benefits."

# Query 5
query = "I am an unemployed youth searching for skill development and employment schemes."

In [28]:
# Generate query embeddings
query_embedding = model.encode(
    [query],
    convert_to_numpy=True
)

In [29]:
# Search Similar Schemes

k = 5

distances, indices = index.search(query_embedding, k)

In [30]:
# Print Retrieved Schemes

for i, idx in enumerate(indices[0], start=1):

    print("="*80)

    print(f"Rank {i}")

    print("Distance :", distances[0][i-1])

    print("Scheme :", df.iloc[idx]["scheme_name"])

    print("Category :", df.iloc[idx]["schemeCategory"])

    print("Level :", df.iloc[idx]["level"])

    print()

Rank 1
Distance : 0.6620084
Scheme : Financial Assistance To Disabled For Self Employment
Category : Skills & Employment, Social welfare & Empowerment, Business & Entrepreneurship
Level : State

Rank 2
Distance : 0.7250856
Scheme : Swarnima
Category : Banking,Financial Services and Insurance, Women and Child
Level : State

Rank 3
Distance : 0.7337213
Scheme : Financial Assistance To Self-Employment For Trained Disabled Persons
Category : Social welfare & Empowerment, Skills & Employment
Level : State

Rank 4
Distance : 0.76767564
Scheme : Financial Assistance For Aids And Appliances For Disabled Persons
Category : Social welfare & Empowerment
Level : State

Rank 5
Distance : 0.7785024
Scheme : Financial Assistance Scheme for Purchase of Agricultural Land to Scheduled Caste Farmers
Category : Banking,Financial Services and Insurance, Social welfare & Empowerment
Level : State



In [31]:
# Show Complete Details
top_index = indices[0][0]

print(df.iloc[top_index]["details"])

The "Financial Assistance to Disabled for Self Employment" is a scheme by the Department of Social Justice & Special Assistance, Govt. of Maharashtra. The objective of this scheme is to facilitate the self-employment of unemployed disabled persons. Under this scheme, financial assistance is provided to persons with disabilities for self-employment, small-scale business, and agro-based project. Only citizens who are permanent residents of the state of Maharashtra are eligible to apply for this scheme. This scheme is 100% funded by the Govt. of Maharashtra.
